Alignment analysis: when R differs, does P adapt? 

In [28]:
from pathlib import Path
from itertools import combinations
import pandas as pd
import numpy as np

region_names = ["California", "Texas", "Germany", "Turkey", "Egypt", "Vietnam", "Nigeria", "India", "Saudi"]

In [29]:
# ========== 输入 ==========
pp_xlsx_path = Path(
    r"dataset\AA1_first_batch\analysis_b_output\behavior_difference_rate_results.xlsx"
)

# 注意：这里必须是原始 regulation clause-region matrix，不是 R_overall_ratio_matrix.csv
r_matrix_path = Path(r"dataset\regions_regulation1.xlsx")

input_folder = Path(
    r"dataset\AA1_first_batch\out_openai_sematic_geodiff_txt"
)

output_file = Path(
    r"dataset\AA1_first_batch\analysis_b_output\behavior_alignment_results.xlsx"
)

output_file.parent.mkdir(parents=True, exist_ok=True)


# ========== 读取有效 app 名 ==========
xls = pd.ExcelFile(pp_xlsx_path)

exclude_sheets = {"average_diff_rate"}

valid_app_names = {
    sheet for sheet in xls.sheet_names
    if sheet not in exclude_sheets
}

print(f"Valid apps from workbook sheets: {len(valid_app_names)}")

Valid apps from workbook sheets: 53


In [30]:
# ========== 读取 regulation matrix ==========
def read_regulation_matrix(path):
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        df = pd.read_excel(path)

    df.columns = df.columns.astype(str).str.strip()

    if "Clause" not in df.columns:
        raise ValueError(
            "Regulation matrix must contain a 'Clause' column. "
            "Do not use R_overall_ratio_matrix.csv here, because it has already aggregated clause-level information."
        )

    df["Clause"] = df["Clause"].astype(str).str.strip()

    region_cols = [c for c in df.columns if c != "Clause"]
    df[region_cols] = df[region_cols].apply(pd.to_numeric, errors="coerce")

    return df


reg_df = read_regulation_matrix(r_matrix_path)
# 方法 1：直接删除并重新赋值（推荐，更符合 Pandas 最佳实践）
reg_df = reg_df.drop(columns=['Bangladesh', 'Pakistan'])
reg_regions = [c for c in reg_df.columns if c != "Clause"]

print(f"Regulation regions: {len(reg_regions)}")
print(reg_regions)

Regulation regions: 9
['California', 'Texas', 'Germany', 'Turkey', 'Egypt', 'Vietnam', 'Nigeria', 'India', 'Saudi']


In [31]:
# ========== 单个 app alignment ==========
def compute_alignment_for_app(app_name, policy_df, reg_df):
    policy_df = policy_df.copy()
    policy_df.columns = policy_df.columns.astype(str).str.strip()

    if "Clause" not in policy_df.columns:
        raise ValueError("Missing Clause column in policy matrix.")

    policy_df["Clause"] = policy_df["Clause"].astype(str).str.strip()

    policy_regions = [c for c in policy_df.columns if c != "Clause"]

    # 只比较 regulation 和 policy 共有的 regions
    common_regions = [r for r in reg_regions if r in policy_regions]

    if len(common_regions) < 2:
        raise ValueError(f"Not enough common regions. Found: {common_regions}")

    # 按 Clause 对齐
    merged = reg_df[["Clause"] + common_regions].merge(
        policy_df[["Clause"] + common_regions],
        on="Clause",
        how="inner",
        suffixes=("_R", "_P")
    )

    rows = []

    for _, row in merged.iterrows():
        clause = row["Clause"]

        for r1, r2 in combinations(common_regions, 2):
            R1 = row[f"{r1}_R"]
            R2 = row[f"{r2}_R"]
            P1 = row[f"{r1}_P"]
            P2 = row[f"{r2}_P"]

            # 跳过缺失值
            if pd.isna(R1) or pd.isna(R2) or pd.isna(P1) or pd.isna(P2):
                continue

            R_diff = int(R1 != R2)
            P_diff = int(P1 != P2)

            if R_diff == 1 and P_diff == 1:
                label = "TP"
            elif R_diff == 1 and P_diff == 0:
                label = "FN"
            elif R_diff == 0 and P_diff == 1:
                label = "FP"
            else:
                label = "TN"

            rows.append({
                "app": app_name,
                "clause": clause,
                "region_1": r1,
                "region_2": r2,
                "region_pair": f"{r1}-{r2}",
                "R_1": R1,
                "R_2": R2,
                "P_1": P1,
                "P_2": P2,
                "R_diff": R_diff,
                "P_diff": P_diff,
                "alignment_type": label,
            })

    detail_df = pd.DataFrame(rows)

    TP = int((detail_df["alignment_type"] == "TP").sum())
    FN = int((detail_df["alignment_type"] == "FN").sum())
    FP = int((detail_df["alignment_type"] == "FP").sum())
    TN = int((detail_df["alignment_type"] == "TN").sum())

    total = TP + FN + FP + TN

    alignment_rate = (TP + TN) / total if total > 0 else np.nan
    recall = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    precision = TP / (TP + FP) if (TP + FP) > 0 else np.nan
    specificity = TN / (TN + FP) if (TN + FP) > 0 else np.nan

    summary = {
        "app": app_name,
        "n_clauses_aligned": merged.shape[0],
        "n_regions": len(common_regions),
        "n_observations": total,

        "TP_Rdiff1_Pdiff1": TP,
        "FN_Rdiff1_Pdiff0": FN,
        "FP_Rdiff0_Pdiff1": FP,
        "TN_Rdiff0_Pdiff0": TN,

        "alignment_rate": alignment_rate,
        "recall_adaptation_coverage": recall,
        "precision_regulation_explains_policy_diff": precision,
        "specificity_policy_consistency_when_reg_same": specificity,
    }

    return summary, detail_df

In [32]:
# ========== 批量处理 ==========
summaries = []
details = []
skipped_apps = []

for apk_dir in input_folder.iterdir():
    if not apk_dir.is_dir():
        continue

    apk_name = apk_dir.name

    # skip folders that are not APK package names
    if "." not in apk_name:
        print(f"[SKIP] Not an APK folder: {apk_dir}")
        continue
    
    # print(f"\n{apk_name}")
    # 只统计 app_matrices 里出现过的 app
    # if app_name not in valid_app_names:
    #     continue
    for valid_app_name in valid_app_names:
        if valid_app_name not in apk_name:
            continue


    app_files = sorted(apk_dir.glob("*_clause_region_matrix.csv")) 
    # print(f"\nProcessing APK: {apk_name} with {len(app_files)} application files.")


    app_name = app_files[0].stem
    # print(f"\nProcessing: {app_name}")

    try:
        # 读取 A
        policy_df = pd.read_csv(app_files[0])

        policy_df.columns = policy_df.columns.astype(str).str.strip()

        if policy_df.shape[1] != 12:
            skipped_apps.append({
                "app": app_name,
                "reason": f"Expected 12 columns: Clause + 11 regions, got {policy_df.shape[1]}"
            })
            continue

        if "Clause" not in policy_df.columns:
            skipped_apps.append({
                "app": app_name,
                "reason": "Missing Clause column"
            })
            continue

        print(f"Processing {app_name}...")

        summary, detail_df = compute_alignment_for_app(app_name, policy_df, reg_df)

        summaries.append(summary)
        details.append(detail_df)

    except Exception as e:
        skipped_apps.append({
            "app": app_name,
            "reason": f"{type(e).__name__}: {e}"
        })


summary_df = pd.DataFrame(summaries)

detail_df = pd.concat(details, ignore_index=True) if details else pd.DataFrame()

skipped_df = pd.DataFrame(skipped_apps)


# ========== overall summary ==========
if not summary_df.empty:
    overall_df = pd.DataFrame([{
        "n_apps": len(summary_df),
        "mean_alignment_rate": summary_df["alignment_rate"].mean(),
        "median_alignment_rate": summary_df["alignment_rate"].median(),

        "mean_recall": summary_df["recall_adaptation_coverage"].mean(),
        "median_recall": summary_df["recall_adaptation_coverage"].median(),

        "mean_precision": summary_df["precision_regulation_explains_policy_diff"].mean(),
        "median_precision": summary_df["precision_regulation_explains_policy_diff"].median(),

        "mean_specificity": summary_df["specificity_policy_consistency_when_reg_same"].mean(),
        "median_specificity": summary_df["specificity_policy_consistency_when_reg_same"].median(),

        "total_TP": summary_df["TP_Rdiff1_Pdiff1"].sum(),
        "total_FN": summary_df["FN_Rdiff1_Pdiff0"].sum(),
        "total_FP": summary_df["FP_Rdiff0_Pdiff1"].sum(),
        "total_TN": summary_df["TN_Rdiff0_Pdiff0"].sum(),
    }])
else:
    overall_df = pd.DataFrame()


# ========== 输出 ==========
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="app_alignment_summary", index=False)
    overall_df.to_excel(writer, sheet_name="overall_summary", index=False)
    detail_df.to_excel(writer, sheet_name="clause_pair_details", index=False)
    skipped_df.to_excel(writer, sheet_name="skipped_apps", index=False)

print(f"\nSaved to: {output_file}")

if not overall_df.empty:
    print("\nOverall summary:")
    print(overall_df.T)

if not skipped_df.empty:
    print("\nSkipped apps:")
    print(skipped_df)

Processing air.com.spilgames.TrollFaceQuestVideoGames2_clause_region_matrix...
Processing alarm.clock.night.watch.talking_clause_region_matrix...
Processing any.splitscreen_clause_region_matrix...
Processing app.phantom_clause_region_matrix...
Processing at.mobilkom.android.meina1_clause_region_matrix...
Processing authenticator.two.factor.authentication.otp_clause_region_matrix...
Processing azt.azt_clause_region_matrix...
Processing bage.image2pdf_clause_region_matrix...
Processing bg.emag.android_clause_region_matrix...
Processing bloodpressure.bloodpressureapppro.bloodpressureapp_clause_region_matrix...
Processing br.com.gringo_clause_region_matrix...
Processing br.com.mobile2you.appgas.costumer_clause_region_matrix...
Processing br.com.promobit.app_clause_region_matrix...
Processing br.tv.horizonte.android.premierefc_clause_region_matrix...
Processing bwebmedia.cookbakedlasagna_clause_region_matrix...
Processing by.green.tuber_clause_region_matrix...
Processing ca.tsn.mobile.andro

## 先生成所有 app 的逐条 observation，再从其中筛出 TP，分别按 region_pair 和 clause_category 统计数量与比例，最后一起写入 Excel

In [33]:
import pandas as pd
import numpy as np

from pathlib import Path
from itertools import combinations
import re

In [34]:
# ========== 输入 ==========
pp_xlsx_path = Path(
    r"dataset\AA1_first_batch\analysis_b_output\behavior_difference_rate_results.xlsx"
)

# 注意：这里必须是原始 regulation clause-region matrix，不是 R_overall_ratio_matrix.csv
r_matrix_path = Path(r"dataset\regions_regulation1.xlsx")

input_folder  = Path(r"dataset\AA1_first_batch\out_openai_sematic_geodiff_txt")

output_file = Path(
    r"dataset\AA1_first_batch\analysis_b_output\behavior_alignment_results2.xlsx"
)

output_file.parent.mkdir(parents=True, exist_ok=True)

In [35]:
# ========== 读取有效 app 名 ==========
xls = pd.ExcelFile(pp_xlsx_path)

exclude_sheets = {"average_diff_rate"}

valid_app_names = {
    sheet for sheet in xls.sheet_names
    if sheet not in exclude_sheets
}

print(f"Valid apps from workbook sheets: {len(valid_app_names)}")

Valid apps from workbook sheets: 53


In [36]:
# ========== 读取 regulation matrix ==========
def read_regulation_matrix(path):
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        df = pd.read_excel(path)

    df.columns = df.columns.astype(str).str.strip()

    if "Clause" not in df.columns:
        raise ValueError(
            "Regulation matrix must contain a 'Clause' column. "
            "Do not use R_overall_ratio_matrix.csv here, because it has already aggregated clause-level information."
        )

    df["Clause"] = df["Clause"].astype(str).str.strip()

    region_cols = [c for c in df.columns if c != "Clause"]
    df[region_cols] = df[region_cols].apply(pd.to_numeric, errors="coerce")

    return df


reg_df = read_regulation_matrix(r_matrix_path)
# 方法 1：直接删除并重新赋值（推荐，更符合 Pandas 最佳实践）
reg_df = reg_df.drop(columns=['Bangladesh', 'Pakistan'])
reg_regions = [c for c in reg_df.columns if c != "Clause"]

print(f"Regulation regions: {len(reg_regions)}")
print(reg_regions)

Regulation regions: 9
['California', 'Texas', 'Germany', 'Turkey', 'Egypt', 'Vietnam', 'Nigeria', 'India', 'Saudi']


In [37]:
# ========== clause category ==========
def get_clause_category(clause):
    """
    根据 clause 编号划分类别：
    P1-P20  -> Personal Data Practices
    CR1-CR5 -> Controller and Recipient
    C1-C2   -> Consent
    R1-R16  -> Rights /E
    其他    -> Others
    """
    clause = str(clause).strip()

    if re.match(r"^P\d+", clause):
        return "P" # "Personal Data Practices"
    elif re.match(r"^CR\d+", clause):
        return "CR" # "Controller and Recipient"
    elif re.match(r"^C\d+", clause):
        return "C" # "Consent"
    elif re.match(r"^R\d+", clause):
        return "R" # "Rights"
    elif re.match(r"^E\d+", clause):
        return "R" # "Rights"
    else:
        return "O" # "Others"

In [38]:
# ========== 读取单个 policy matrix ==========
def read_policy_matrix(path):
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    elif path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    df.columns = df.columns.astype(str).str.strip()

    if "Clause" not in df.columns:
        raise ValueError(f"Missing Clause column in {path.name}")

    df["Clause"] = df["Clause"].astype(str).str.strip()

    region_cols = [c for c in df.columns if c != "Clause"]
    df[region_cols] = df[region_cols].apply(pd.to_numeric, errors="coerce")

    return df


# ========== 单个 app alignment ==========
def compute_alignment_for_app(app_name, policy_df, reg_df):
    policy_df = policy_df.copy()
    policy_df.columns = policy_df.columns.astype(str).str.strip()

    if "Clause" not in policy_df.columns:
        raise ValueError("Missing Clause column in policy matrix.")

    policy_df["Clause"] = policy_df["Clause"].astype(str).str.strip()

    policy_regions = [c for c in policy_df.columns if c != "Clause"]

    # 只比较 regulation 和 policy 共有的 regions
    common_regions = [r for r in reg_regions if r in policy_regions]

    if len(common_regions) < 2:
        raise ValueError(f"Not enough common regions. Found: {common_regions}")

    # 按 Clause 对齐
    merged = reg_df[["Clause"] + common_regions].merge(
        policy_df[["Clause"] + common_regions],
        on="Clause",
        how="inner",
        suffixes=("_R", "_P")
    )

    rows = []

    for _, row in merged.iterrows():
        clause = row["Clause"]
        clause_category = get_clause_category(clause)

        for r1, r2 in combinations(common_regions, 2):
            R1 = row[f"{r1}_R"]
            R2 = row[f"{r2}_R"]
            P1 = row[f"{r1}_P"]
            P2 = row[f"{r2}_P"]

            # 跳过缺失值
            if pd.isna(R1) or pd.isna(R2) or pd.isna(P1) or pd.isna(P2):
                continue

            R_diff = int(R1 != R2)
            P_diff = int(P1 != P2)

            if R_diff == 1 and P_diff == 1:
                label = "TP"
            elif R_diff == 1 and P_diff == 0:
                label = "FN"
            elif R_diff == 0 and P_diff == 1:
                label = "FP"
            else:
                label = "TN"

            rows.append({
                "app": app_name,
                "clause": clause,
                "clause_category": clause_category,
                "region_1": r1,
                "region_2": r2,
                "region_pair": f"{r1}-{r2}",
                "R_1": R1,
                "R_2": R2,
                "P_1": P1,
                "P_2": P2,
                "R_diff": R_diff,
                "P_diff": P_diff,
                "alignment_type": label,
            })

    detail_df = pd.DataFrame(rows)

    if detail_df.empty:
        summary = {
            "app": app_name,
            "n_clauses_aligned": merged.shape[0],
            "n_regions": len(common_regions),
            "n_observations": 0,

            "TP_Rdiff1_Pdiff1": 0,
            "FN_Rdiff1_Pdiff0": 0,
            "FP_Rdiff0_Pdiff1": 0,
            "TN_Rdiff0_Pdiff0": 0,

            "alignment_rate": np.nan,
            "recall_adaptation_coverage": np.nan,
            "precision_regulation_explains_policy_diff": np.nan,
            "specificity_policy_consistency_when_reg_same": np.nan,
        }

        return summary, detail_df

    TP = int((detail_df["alignment_type"] == "TP").sum())
    FN = int((detail_df["alignment_type"] == "FN").sum())
    FP = int((detail_df["alignment_type"] == "FP").sum())
    TN = int((detail_df["alignment_type"] == "TN").sum())

    total = TP + FN + FP + TN

    alignment_rate = (TP + TN) / total if total > 0 else np.nan
    recall = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    precision = TP / (TP + FP) if (TP + FP) > 0 else np.nan
    specificity = TN / (TN + FP) if (TN + FP) > 0 else np.nan

    summary = {
        "app": app_name,
        "n_clauses_aligned": merged.shape[0],
        "n_regions": len(common_regions),
        "n_observations": total,

        "TP_Rdiff1_Pdiff1": TP,
        "FN_Rdiff1_Pdiff0": FN,
        "FP_Rdiff0_Pdiff1": FP,
        "TN_Rdiff0_Pdiff0": TN,

        "alignment_rate": alignment_rate,
        "recall_adaptation_coverage": recall,
        "precision_regulation_explains_policy_diff": precision,
        "specificity_policy_consistency_when_reg_same": specificity,
    }

    return summary, detail_df


# ========== 批量处理所有 app ==========
all_summary_rows = []
all_detail_dfs = []

skipped_files = []


for apk_dir in input_folder.iterdir():
    if not apk_dir.is_dir():
        continue

    apk_name = apk_dir.name

    # skip folders that are not APK package names
    if "." not in apk_name:
        print(f"[SKIP] Not an APK folder: {apk_dir}")
        continue
    
    # print(f"\n{apk_name}")
    # 只统计 app_matrices 里出现过的 app
    # if app_name not in valid_app_names:
    #     continue
    for valid_app_name in valid_app_names:
        if valid_app_name not in apk_name:
            continue


    app_files = sorted(apk_dir.glob("*_clause_region_matrix.csv")) 
    # print(f"\nProcessing APK: {apk_name} with {len(app_files)} application files.")


    app_name = app_files[0].stem
    # print(f"\nProcessing: {app_name}")

    try:
        # 读取 A
        policy_df = pd.read_csv(app_files[0])


        # 如果你仍然希望跳过 region 列不是 12 的 matrix，可以保留这段
        # 如果不需要这个限制，可以删掉
        region_cols = [c for c in policy_df.columns if c != "Clause"]
        # if len(region_cols) != 12:
        #     skipped_files.append({
        #         "file": file_path.name,
        #         "reason": f"region column count is {len(region_cols)}, not 12"
        #     })
        #     continue

        summary, detail_df = compute_alignment_for_app(
            app_name=app_name,
            policy_df=policy_df,
            reg_df=reg_df
        )

        all_summary_rows.append(summary)

        if not detail_df.empty:
            all_detail_dfs.append(detail_df)

        print(f"Processed: {app_name}")

    except Exception as e:
        skipped_files.append({
            "file": app_name,
            "reason": str(e)
        })
        print(f"Skipped {app_name}: {e}")


app_summary_df = pd.DataFrame(all_summary_rows)

if len(all_detail_dfs) > 0:
    all_details_df = pd.concat(all_detail_dfs, ignore_index=True)
else:
    all_details_df = pd.DataFrame()

skipped_df = pd.DataFrame(skipped_files)

print(f"Processed apps: {len(app_summary_df)}")
print(f"Total observations: {len(all_details_df)}")
print(f"Skipped files: {len(skipped_df)}")


# ========== 汇总 TP ==========
if not all_details_df.empty:
    tp_df = all_details_df[all_details_df["alignment_type"] == "TP"].copy()

    # 每个 region pair 的 TP 数量
    TP_by_region_pair = (
        tp_df
        .groupby("region_pair")
        .size()
        .reset_index(name="TP_count")
        .sort_values("TP_count", ascending=False)
    )

    # 每个 region pair 的总 observation 和 TP ratio
    region_pair_total = (
        all_details_df
        .groupby("region_pair")
        .size()
        .reset_index(name="total_observations")
    )

    TP_by_region_pair = TP_by_region_pair.merge(
        region_pair_total,
        on="region_pair",
        how="right"
    )

    TP_by_region_pair["TP_count"] = TP_by_region_pair["TP_count"].fillna(0).astype(int)
    TP_by_region_pair["TP_rate_among_all_observations"] = (
        TP_by_region_pair["TP_count"] / TP_by_region_pair["total_observations"]
    )

    TP_by_region_pair = TP_by_region_pair.sort_values(
        "TP_count",
        ascending=False
    )

    # 每个 clause category 的 TP 数量
    TP_by_clause_category = (
        tp_df
        .groupby("clause_category")
        .size()
        .reset_index(name="TP_count")
        .sort_values("TP_count", ascending=False)
    )

    category_total = (
        all_details_df
        .groupby("clause_category")
        .size()
        .reset_index(name="total_observations")
    )

    TP_by_clause_category = TP_by_clause_category.merge(
        category_total,
        on="clause_category",
        how="right"
    )

    TP_by_clause_category["TP_count"] = TP_by_clause_category["TP_count"].fillna(0).astype(int)
    TP_by_clause_category["TP_rate_among_all_observations"] = (
        TP_by_clause_category["TP_count"] / TP_by_clause_category["total_observations"]
    )

    TP_by_clause_category = TP_by_clause_category.sort_values(
        "TP_count",
        ascending=False
    )

    # region pair × clause category 的 TP 数量
    TP_by_region_pair_and_category = (
        tp_df
        .groupby(["region_pair", "clause_category"])
        .size()
        .reset_index(name="TP_count")
        .sort_values(["region_pair", "TP_count"], ascending=[True, False])
    )

    # 交叉表形式，更适合画热力图
    TP_region_category_pivot = (
        TP_by_region_pair_and_category
        .pivot(index="region_pair", columns="clause_category", values="TP_count")
        .fillna(0)
        .astype(int)
    )

    # 每个具体 clause 的 TP 数量
    TP_by_clause = (
        tp_df
        .groupby(["clause_category", "clause"])
        .size()
        .reset_index(name="TP_count")
        .sort_values("TP_count", ascending=False)
    )

    clause_total = (
        all_details_df
        .groupby(["clause_category", "clause"])
        .size()
        .reset_index(name="total_observations")
    )

    TP_by_clause = TP_by_clause.merge(
        clause_total,
        on=["clause_category", "clause"],
        how="right"
    )

    TP_by_clause["TP_count"] = TP_by_clause["TP_count"].fillna(0).astype(int)
    TP_by_clause["TP_rate_among_all_observations"] = (
        TP_by_clause["TP_count"] / TP_by_clause["total_observations"]
    )

    TP_by_clause = TP_by_clause.sort_values(
        "TP_count",
        ascending=False
    )

    # 每个 app × region pair 的 TP 数量
    TP_by_app_region_pair = (
        tp_df
        .groupby(["app", "region_pair"])
        .size()
        .reset_index(name="TP_count")
        .sort_values(["app", "TP_count"], ascending=[True, False])
    )

    # 每个 app × clause category 的 TP 数量
    TP_by_app_category = (
        tp_df
        .groupby(["app", "clause_category"])
        .size()
        .reset_index(name="TP_count")
        .sort_values(["app", "TP_count"], ascending=[True, False])
    )

    # 可选：统计所有 alignment 类型在 region pair 上的分布
    alignment_by_region_pair = (
        all_details_df
        .groupby(["region_pair", "alignment_type"])
        .size()
        .reset_index(name="count")
        .pivot(index="region_pair", columns="alignment_type", values="count")
        .fillna(0)
        .astype(int)
        .reset_index()
    )

    # 可选：统计所有 alignment 类型在 clause category 上的分布
    alignment_by_clause_category = (
        all_details_df
        .groupby(["clause_category", "alignment_type"])
        .size()
        .reset_index(name="count")
        .pivot(index="clause_category", columns="alignment_type", values="count")
        .fillna(0)
        .astype(int)
        .reset_index()
    )

else:
    tp_df = pd.DataFrame()
    TP_by_region_pair = pd.DataFrame()
    TP_by_clause_category = pd.DataFrame()
    TP_by_region_pair_and_category = pd.DataFrame()
    TP_region_category_pivot = pd.DataFrame()
    TP_by_clause = pd.DataFrame()
    TP_by_app_region_pair = pd.DataFrame()
    TP_by_app_category = pd.DataFrame()
    alignment_by_region_pair = pd.DataFrame()
    alignment_by_clause_category = pd.DataFrame()


# ========== 写入 Excel ==========
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    app_summary_df.to_excel(writer, sheet_name="app_summary", index=False)

    all_details_df.to_excel(writer, sheet_name="all_details", index=False)

    tp_df.to_excel(writer, sheet_name="TP_details", index=False)

    TP_by_region_pair.to_excel(writer, sheet_name="TP_by_region_pair", index=False)

    TP_by_clause_category.to_excel(writer, sheet_name="TP_by_clause_category", index=False)

    TP_by_region_pair_and_category.to_excel(
        writer,
        sheet_name="TP_region_category_long",
        index=False
    )

    TP_region_category_pivot.to_excel(
        writer,
        sheet_name="TP_region_category_pivot"
    )

    TP_by_clause.to_excel(writer, sheet_name="TP_by_clause", index=False)

    TP_by_app_region_pair.to_excel(
        writer,
        sheet_name="TP_by_app_region_pair",
        index=False
    )

    TP_by_app_category.to_excel(
        writer,
        sheet_name="TP_by_app_category",
        index=False
    )

    alignment_by_region_pair.to_excel(
        writer,
        sheet_name="align_by_region_pair",
        index=False
    )

    alignment_by_clause_category.to_excel(
        writer,
        sheet_name="align_by_category",
        index=False
    )

    skipped_df.to_excel(writer, sheet_name="skipped_files", index=False)

print(f"Saved alignment results to: {output_file}")

Processed: air.com.spilgames.TrollFaceQuestVideoGames2_clause_region_matrix
Processed: alarm.clock.night.watch.talking_clause_region_matrix
Processed: any.splitscreen_clause_region_matrix
Processed: app.phantom_clause_region_matrix
Processed: at.mobilkom.android.meina1_clause_region_matrix
Processed: authenticator.two.factor.authentication.otp_clause_region_matrix
Processed: azt.azt_clause_region_matrix
Processed: bage.image2pdf_clause_region_matrix
Processed: bg.emag.android_clause_region_matrix
Processed: bloodpressure.bloodpressureapppro.bloodpressureapp_clause_region_matrix
Processed: br.com.gringo_clause_region_matrix
Processed: br.com.mobile2you.appgas.costumer_clause_region_matrix
Processed: br.com.promobit.app_clause_region_matrix
Processed: br.tv.horizonte.android.premierefc_clause_region_matrix
Processed: bwebmedia.cookbakedlasagna_clause_region_matrix
Processed: by.green.tuber_clause_region_matrix
Processed: ca.tsn.mobile.android_clause_region_matrix
Processed: cast.video.sc

## 生成混淆矩阵

In [41]:
import pandas as pd
import numpy as np
from pathlib import Path

region_order = ["California", "Texas", "Germany", "Turkey", "Egypt", "Vietnam", "Nigeria", "India", "Saudi"]
# ========== 输入 ==========
input_xlsx = Path(
    r"dataset\AA1_first_batch\analysis_b_output\behavior_alignment_results2.xlsx"
)
output_xlsx = Path(
    r"dataset\AA1_first_batch\analysis_b_output\behavior_alignment_results_with_TP_by_region_pair.xlsx" #PP_TP_by_region_pair
)

sheet_name = "TP_by_region_pair"

# ========== 读取 TP_by_region_pair ==========
df = pd.read_excel(input_xlsx, sheet_name=sheet_name)

df.columns = df.columns.astype(str).str.strip()

required_cols = {"region_pair", "TP_count"}
missing_cols = required_cols - set(df.columns)

if missing_cols:
    raise ValueError(f"Missing columns in {sheet_name}: {missing_cols}")

df["region_pair"] = df["region_pair"].astype(str).str.strip()
df["TP_count"] = pd.to_numeric(df["TP_count"], errors="coerce").fillna(0).astype(int)


# ========== 拆分 region pair ==========
def split_region_pair(pair):
    parts = str(pair).split("-")
    if len(parts) != 2:
        raise ValueError(f"Invalid region_pair format: {pair}")
    return parts[0].strip(), parts[1].strip()


df[["region_1", "region_2"]] = df["region_pair"].apply(
    lambda x: pd.Series(split_region_pair(x))
)


# ========== 自动获取所有 region ==========
all_regions = set(df["region_1"]).union(set(df["region_2"]))

regions = [r for r in region_order if r in all_regions]

print(f"Number of regions: {len(regions)}")
print(regions)


# ========== 构建 TP count 矩阵 ==========
tp_matrix = pd.DataFrame(
    data=0,
    index=regions,
    columns=regions
)

for _, row in df.iterrows():
    r1 = row["region_1"]
    r2 = row["region_2"]
    tp = row["TP_count"]

    tp_matrix.loc[r1, r2] = tp
    tp_matrix.loc[r2, r1] = tp

# 对角线设为 0
for r in tp_matrix.index:
    tp_matrix.loc[r, r] = 0

print(tp_matrix)

tp_matrix_path= r"dataset\AA1_first_batch\analysis_b_output\B_TP_by_region_pair.csv" #
tp_matrix.to_csv(tp_matrix_path, index=True)


# ========== 可选：构建 TP rate 矩阵 ==========
# 如果你的表里有 TP_rate_among_all_observations，也可以一起转成矩阵
if "TP_rate_among_all_observations" in df.columns:
    df["TP_rate_among_all_observations"] = pd.to_numeric(
        df["TP_rate_among_all_observations"],
        errors="coerce"
    ).fillna(0)

    tp_rate_matrix = pd.DataFrame(
        data=0.0,
        index=regions,
        columns=regions
    )

    for _, row in df.iterrows():
        r1 = row["region_1"]
        r2 = row["region_2"]
        rate = row["TP_rate_among_all_observations"]

        tp_rate_matrix.loc[r1, r2] = rate
        tp_rate_matrix.loc[r2, r1] = rate

for r in tp_matrix.index:
    tp_matrix.loc[r, r] = 0

else:
    tp_rate_matrix = None


# ========== 写回 Excel ==========
with pd.ExcelWriter(
    output_xlsx,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    tp_matrix.to_excel(writer, sheet_name="TP_region_pair_matrix")

    if tp_rate_matrix is not None:
        tp_rate_matrix.to_excel(writer, sheet_name="TP_region_pair_rate_matrix")

print(f"Saved TP region-pair matrix to: {output_xlsx}")

Number of regions: 9
['California', 'Texas', 'Germany', 'Turkey', 'Egypt', 'Vietnam', 'Nigeria', 'India', 'Saudi']
            California  Texas  Germany  Turkey  Egypt  Vietnam  Nigeria  \
California           0      0       26      28     24       40       15   
Texas                0      0       10      21     11       26        0   
Germany             26     10        0      23      2        6        6   
Turkey              28     21       23       0     24       35       22   
Egypt               24     11        2      24      0        5        6   
Vietnam             40     26        6      35      5        0       19   
Nigeria             15      0        6      22      6       19        0   
India               35     17        1      27      0       11       20   
Saudi               35     18        9      33     12       22        8   

            India  Saudi  
California     35     35  
Texas          17     18  
Germany         1      9  
Turkey         27     33  

don't need

Do larger regulation differences increase the likelihood that privacy policies become region-specific?

input: 从dataset\AA1_first_batch\AA2_second_100_batch\analysis_pp_output\privacy_policy_difference_rate_results.xlsx读取 app sheet 名字，只统计这些 app 对应的 CSV。

In [8]:
from pathlib import Path
from itertools import combinations
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, fisher_exact
# import statsmodels.api as sm

In [ ]:
# ========== 输入路径 ==========
pp_xlsx_path = Path(r"dataset\AA1_first_batch\analysis_b_output\behavior_difference_rate_results.xlsx")

# 这里换成你的 regulation difference matrix 路径
# 可以是 csv，也可以是 xlsx
r_matrix_path = Path(r"dataset\out_R\R_overall_ratio_matrix.csv")

output_path = Path(r"dataset\AA1_first_batch\analysis_b_output\behavior_regulation_association_results.xlsx")


In [13]:
def read_matrix(path, sheet_name=None):
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path, index_col=0)
    else:
        df = pd.read_excel(path, sheet_name=sheet_name, index_col=0)
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()
    return df.apply(pd.to_numeric, errors="coerce")

def flatten_upper_triangle(df, value_name):
    regions = list(df.index)
    rows = []
    for i in range(len(regions)):
        for j in range(i + 1, len(regions)):
            r1, r2 = regions[i], regions[j]
            if r1 in df.columns and r2 in df.columns:
                rows.append({
                    "region_1": r1,
                    "region_2": r2,
                    "region_pair": f"{r1}-{r2}",
                    value_name: df.loc[r1, r2]
                })
    return pd.DataFrame(rows)

def safe_spearman(x, y):
    if x.nunique() < 2 or y.nunique() < 2:
        return np.nan, np.nan
    return spearmanr(x, y)

def calc_2x2(df, r_threshold):
    tmp = df.copy()
    tmp["R_high"] = (tmp["R_diff"] >= r_threshold).astype(int)
    tmp["PolicyDiff"] = (tmp["P_diff"] > 0).astype(int)

    # rows: R_high 1/0, columns: PolicyDiff 1/0
    a = int(((tmp["R_high"] == 1) & (tmp["PolicyDiff"] == 1)).sum())
    b = int(((tmp["R_high"] == 1) & (tmp["PolicyDiff"] == 0)).sum())
    c = int(((tmp["R_high"] == 0) & (tmp["PolicyDiff"] == 1)).sum())
    d = int(((tmp["R_high"] == 0) & (tmp["PolicyDiff"] == 0)).sum())

    # Fisher exact test table: [[a,b],[c,d]]
    try:
        odds_ratio, p_value = fisher_exact([[a, b], [c, d]])
    except Exception:
        odds_ratio, p_value = np.nan, np.nan

    return a, b, c, d, odds_ratio, p_value

R_matrix = read_matrix(r_matrix_path)
R_long = flatten_upper_triangle(R_matrix, "R_diff")

r_threshold = R_long["R_diff"].quantile(0.75)

xls = pd.ExcelFile(pp_xlsx_path)
app_sheets = [s for s in xls.sheet_names if s != "average_diff_rate"]

all_pairs = []
results = []

for sheet in app_sheets:
    P_matrix = read_matrix(pp_xlsx_path, sheet_name=sheet)
    P_long = flatten_upper_triangle(P_matrix, "P_diff")

    merged = R_long.merge(P_long[["region_pair", "P_diff"]], on="region_pair", how="inner")
    merged["app"] = sheet
    merged["PolicyDiff"] = (merged["P_diff"] > 0).astype(int)

    rho, sp_p = safe_spearman(merged["R_diff"], merged["P_diff"])
    a, b, c, d, or_value, fisher_p = calc_2x2(merged, r_threshold)

    results.append({
        "app": sheet,
        "n_region_pairs": len(merged),
        "n_policy_diff": int(merged["PolicyDiff"].sum()),
        "n_policy_no_diff": int((merged["PolicyDiff"] == 0).sum()),
        "policy_diff_rate": merged["PolicyDiff"].mean(),

        "spearman_rho": rho,
        "spearman_p": sp_p,

        "R_high_threshold": r_threshold,
        "high_R_policy_diff": a,
        "high_R_policy_no_diff": b,
        "low_R_policy_diff": c,
        "low_R_policy_no_diff": d,
        "odds_ratio": or_value,
        "fisher_p": fisher_p,
    })

    all_pairs.append(merged)

In [14]:
pair_df = pd.concat(all_pairs, ignore_index=True)
result_df = pd.DataFrame(results)

# pooled 2x2
a, b, c, d, pooled_or, pooled_fisher_p = calc_2x2(pair_df, r_threshold)

pooled_df = pd.DataFrame([{
    "n_apps": len(app_sheets),
    "n_rows": len(pair_df),
    "R_high_threshold": r_threshold,
    "high_R_policy_diff": a,
    "high_R_policy_no_diff": b,
    "low_R_policy_diff": c,
    "low_R_policy_no_diff": d,
    "pooled_odds_ratio": pooled_or,
    "pooled_fisher_p": pooled_fisher_p,
}])

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    pair_df.to_excel(writer, sheet_name="pair_level_data", index=False)
    result_df.to_excel(writer, sheet_name="app_level_results", index=False)
    pooled_df.to_excel(writer, sheet_name="pooled_2x2", index=False)

print(f"Saved to: {output_path}")
print(pooled_df.T)

Saved to: dataset\AA1_first_batch\AA2_second_100_batch\analysis_pp_output\pp_regulation_association_results.xlsx
                                0
n_apps                  14.000000
n_rows                 503.000000
R_high_threshold         0.347826
high_R_policy_diff      71.000000
high_R_policy_no_diff  110.000000
low_R_policy_diff      123.000000
low_R_policy_no_diff   199.000000
pooled_odds_ratio        1.044272
pooled_fisher_p          0.848884
